<a href="https://colab.research.google.com/github/mwanginjuguna/regenerative-agriculture-chatbot/blob/main/regen_agri_chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chat with Regenerative Agriculture Docs

Author: Francis M. Njuguna

Description: A simple AI application for regenerative agriculture farmers and enthusiasts to chat with an AI that is specialized and optimized for all things regenerative farming. This is a regenerative agriculture AI expert.

Technologies: LlammaIndex(orchestration framework), all-MiniLM-L6-v2(embedding model), ChromaDB (Vector db), Gemini via Google AI Studio API (LLM).

Data: Quality documents from online sources about regenerative farming. Cited in references at bottom of this notebook.

## **Setting up & Organizing documents with metadata**

---
| **Metadata Field** | Value |
|---|---|
| source file | document 1 |
| topic | topic 1 |
| year | 2025 |
---


#### Setup the required libraries and initialize files

We are going to setup a folder that will store the agriculture pdfs.

In addition, we will setup the LLM API keys in the environment. In this case, we are using gemini ai studio api keys.

In [1]:
# Install necessary libraries
!pip install -qU llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface chromadb pypdf llama-index-llms-gemini

# For LlamaIndex to easily grab PDFs
import os
import requests

# 1. Setup
!mkdir -p 'data/agri_docs'


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.9/328.9 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta

##### PDF files prep
These pdf documents are handbooks, reports, and journal article freely-available and collected from online sources. They all focus on the topic of regenerative agriculture.

We will download them to the local folder environment for use in this task.

In [2]:
# a list of pdfs to download
pdf_urls = [
    "https://www.leopold.iastate.edu/files/pubs-and-papers/2008-06-it-starts-soil-and-organic-agriculture-can-help.pdf",
    "https://archive.org/stream/in.ernet.dli.2015.270767/2015.270767.An-Agricultural_djvu.txt",
    "https://orgprints.org/id/eprint/37226/1/myth_of_the_peasant_in_the_global_organic_farming_movement.pdf",
    "https://rodaleinstitute.org/wp-content/uploads/Rodale-Soil-Carbon-White-Paper_v8.pdf",
    "https://avrdc.org/download/project-support/v4pp/training-trainers/1-5-GAP/Regenerative-Agriculture-Handbook.pdf",
    "https://livelihoods.eu/wp-content/uploads/2024/07/Nov_2023_Handbook-to-Regenerative-Agriculture.pdf",
    "https://www.soilheroesfoundation.com/wp-content/uploads/2022/12/Guidebook-for-regenerative-farming.pdf",
    "https://www.nestle.com/sites/default/files/2025-06/sugarcane-handbook-regenerative-agriculture-2025.pdf",
    "https://www.unilever.co.uk/files/6214b484-7875-4042-90ed-30122b2bbf05/regenerative-agriculture-principles-and-implementation-guide-april-2021.pdf",
    "https://www.fao.org/fileadmin/templates/nr/sustainability_pathways/docs/Compilation_techniques_organic_agriculture_rev.pdf",
    "https://extension.okstate.edu/fact-sheets/print-publications/afs/regenerative-agriculture-an-introduction-and-overview-afs-9412-a.pdf",
    "https://www.nature.org/content/dam/tnc/nature/en/documents/nature-lab-lesson-plans/TeachingGuide_RegenerativeAg.pdf",
    "https://www.saa-safe.org/elfiles/xc85X4Fu/Size%20adjusted_FIN%20-%20web%20compressed%20-%20Basket%20of%20RA%20Tech%202024.pdf",
    "https://unfccc.int/sites/default/files/resource/RegenAg.pdf",
    "https://www.mdpi.com/2071-1050/15/22/15941/pdf?version=1700039494",
    "https://www.incda-fundulea.ro/new4/images/rar/nr41fol/rar41.33.pdf",
    "https://www.cambridge.org/core/services/aop-cambridge-core/content/view/03E14A6403C7CDBF7472100F1347C6C1/S0962728623000283a.pdf",
    "https://www.sciencedirect.com/science/article/pii/S2211912420300584",
    "https://www.zanmiparis.org/wp-content/uploads/2010/03/McClintock_ZLP_English.pdf"
]

# download each of the pdfs
for url in pdf_urls:
    file_path = "data/agri_docs/" + url.split("/")[-1]
    response = requests.get(url)
    with open(file_path, "wb") as f:
        f.write(response.content)
    print("Downloaded: " + file_path)

print("Setup complete. All PDFs downloaded.")

Downloaded: data/agri_docs/2008-06-it-starts-soil-and-organic-agriculture-can-help.pdf
Downloaded: data/agri_docs/2015.270767.An-Agricultural_djvu.txt
Downloaded: data/agri_docs/myth_of_the_peasant_in_the_global_organic_farming_movement.pdf
Downloaded: data/agri_docs/Rodale-Soil-Carbon-White-Paper_v8.pdf
Downloaded: data/agri_docs/Regenerative-Agriculture-Handbook.pdf
Downloaded: data/agri_docs/Nov_2023_Handbook-to-Regenerative-Agriculture.pdf
Downloaded: data/agri_docs/Guidebook-for-regenerative-farming.pdf
Downloaded: data/agri_docs/sugarcane-handbook-regenerative-agriculture-2025.pdf
Downloaded: data/agri_docs/regenerative-agriculture-principles-and-implementation-guide-april-2021.pdf
Downloaded: data/agri_docs/Compilation_techniques_organic_agriculture_rev.pdf
Downloaded: data/agri_docs/regenerative-agriculture-an-introduction-and-overview-afs-9412-a.pdf
Downloaded: data/agri_docs/TeachingGuide_RegenerativeAg.pdf
Downloaded: data/agri_docs/Size%20adjusted_FIN%20-%20web%20compressed

### RAG Indexing Code

- Loading, Chunking, Embedding, and Indexing

### 1. Configure the Core Components

**A. Set the Embedding Model (The Translator)**

We use the `sentence-transformers/all-MiniLM-L6-v2`, a fast, high-quality open-source model suitable for Colab for embeddings.


In [3]:
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding Model set:", Settings.embed_model.model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model set: sentence-transformers/all-MiniLM-L6-v2


**B. Configure the Vector Database (The Memory)**

Next, we create a persistent vector client, which saves your vectors to the disk.


In [4]:
CHROMA_PATH = "./agri_chroma_db"
db = chromadb.PersistentClient(path=CHROMA_PATH)

# A collection is like a table; it holds your vectors for this project.
chroma_collection = db.get_or_create_collection("regenerative_agri_index")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Link the vector store to LlamaIndex
storage_context = StorageContext.from_defaults(vector_store=vector_store)
print("ChromaDB Client initialized at:", CHROMA_PATH)


ChromaDB Client initialized at: ./agri_chroma_db


#### Configuring the model

We need to import the necessary libraries and retrieve our API key from Colab's `userdata` secrets.

In [6]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API configured successfully!")

Gemini API configured successfully!


Next, we initialize the Generative Model to use, in this case `gemini-2.5-flash-lite` model.

In [9]:
model = genai.GenerativeModel('gemini-2.5-flash-lite')

print("Gemini model initialized!")

Gemini model initialized!


Let's test the model by asking it to generate a poem.

In [10]:
prompt = "Write a short poem about the beauty of regenerative agriculture."
response = model.generate_content(prompt)

print("\n--- Generated Content ---")
print(response.text)


--- Generated Content ---
The soil, a canvas, dark and deep,
Where sleepy seeds begin to leap.
No barren fields, but life reborn,
With every dewdrop, every morn.

The roots embrace, a hidden hold,
Protecting earth from winds so bold.
And tiny creatures, swift and low,
Make fertile ground where good things grow.

The air breathes clean, the waters pure,
A gentle promise, to endure.
The land rejoices, rich and free,
A vibrant, living tapestry.


**C. Use the Configured the LLM (The Brain)**

We are set the LLM to use an API key for Gemini


In [14]:
!pip install -qU llama-index-llms-gemini

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 52.2 MB/s eta 0:00:00


In [17]:
from llama_index.llms.gemini import Gemini

Settings.llm = Gemini(model_name='gemini-2.0-flash', api_key=GOOGLE_API_KEY)
print("LLM set with Gemini model: gemini-2.0-flash")

/tmp/ipython-input-4188311852.py:5: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/)
  Settings.llm = Gemini(model_name='gemini-2.0-flash', api_key=GOOGLE_API_KEY)


LLM set with Gemini model: gemini-2.0-flash


### 2. Load and Chunk the Documents

We will configure the reader to find and load all documents in the 'data/agri_docs' folder.

The LlamaIndex's SimpleDirectoryReader will also handle the initial document parsing (PDF to text).


In [18]:

documents = SimpleDirectoryReader("data/agri_docs").load_data()
print(f"Loaded {len(documents)} document(s).")


Failed to load file /content/data/agri_docs/RegenAg.pdf with error: RetryError[<Future at 0x7dbe2aec5b20 state=finished raised PdfStreamError>]. Skipping...
Loaded 748 document(s).


### 3. Index Creation (Chunking & Embedding)

This step orchestrates everything:

1. It chunks the documents (LlamaIndex uses intelligent chunking by default).
2. It sends each chunk to the `Settings.embed_model` to create a vector.
3. It stores the vector and the original text chunk in `storage_context` (ChromaDB).


In [19]:
print("Starting Index Creation (Chunking and Embedding)...")
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context
)
print("Indexing Complete! All documents are now embedded and stored in ChromaDB.")



Starting Index Creation (Chunking and Embedding)...
Indexing Complete! All documents are now embedded and stored in ChromaDB.


### 4. Testing Retrieval ---
**Finally, does it work?**

To test if the embeddings work, we ask a question related to what we hope the embeddings have in store.

This should work since Settings.llm is configured with an actual LLM service - Gemini.


In [20]:

try:
    query_engine = index.as_query_engine()
    response = query_engine.query("What are the core principles of regenerative agriculture in Africa, and who are the beneficiaries?")
    print("\n--- RAG Response ---")
    print(response)

except AttributeError:
    print("\n--- RAG Test (No LLM Configured) ---")
    print("Indexing was successful. To get a full answer, connect Settings.llm to an LLM like Gemini or GPT-4.")


--- RAG Response ---
There are eleven core principles that govern regenerative farming in Africa. Agricultural practitioners, researchers, and policymakers all have a crucial and valued role in implementing these principles.



Let's do another question.

In [21]:
response = query_engine.query("Explain in details, the core principles of regenerative agriculture.")
print("\n--- Response ---")
print(response)


--- Response ---
Regenerative agriculture operates on core principles that include enhancing agroecosystems (soil, water, and biodiversity), creating context-specific designs and holistic decisions tailored to each farm, ensuring equitable relationships among all involved, and fostering continuous growth for individuals, farms, and communities.



Let's try something we did not probably put in the context.

In [22]:
response = query_engine.query("What spacecraft first landed on Venus and when?")
print("\n--- Wrong question ---")
print(response)


--- Wrong question ---
I'm sorry, but this question cannot be answered from the given source.

